In [ ]:
import pandas as pd
from transformers import pipeline
from transformers import AutoTokenizer, AutoModelForTokenClassification

In [ ]:
MODEL_NAME = "d4data/biomedical-ner-all"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForTokenClassification.from_pretrained(MODEL_NAME)

In [ ]:
def clean_corpus(corpus):
    cleaned_corpus = corpus.replace(".", " ")
    return cleaned_corpus

In [ ]:
import re
def extract_age(value):
    match = re.search(r"\d+", value)
    return int(match.group()) if match else None

In [ ]:
def aggregate_entities(df):
    result = {}

    for group, subdf in df.groupby("entity_group"):
        values = subdf["value"].tolist()

        if group == "Age":
            # take first valid age
            ages = [extract_age(v) for v in values if extract_age(v) is not None]
            result[group] = ages[0] if ages else None

        elif group == "Sex":
            result[group] = values[0] if values else None

        else:
            # remove duplicates while preserving order
            seen = set()
            unique_values = []
            for v in values:
                if v not in seen:
                    seen.add(v)
                    unique_values.append(v)

            result[group] = unique_values

    return pd.DataFrame([result])

In [ ]:
def ner_prediction(corpus):
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForTokenClassification.from_pretrained(MODEL_NAME)

    ner_pipeline = pipeline(
        "token-classification",
        model=model,
        tokenizer=tokenizer,
        aggregation_strategy="max",
    )

    cleaned_corpus = clean_corpus(corpus)

    pred = ner_pipeline(cleaned_corpus)

    pred_df = pd.DataFrame(pred)
    
    actual_values_list = []
    for _, pred_df_row in pred_df.iterrows():
        actual_word = corpus[pred_df_row['start']: pred_df_row['end']]
        actual_values_list.append(actual_word)

    pred_df['value'] = actual_values_list
    
    if len(pred_df) != 0:
        pred_df = pred_df[['entity_group', 'value', 'word', 'start', 'end', 'score']]
        pred_df = pred_df.rename(columns={"Sign_symptom": "Symptoms"})
        pred_df = pred_df.drop_duplicates(
            subset = ['entity_group', 'value'],
            keep = 'first'
    ).reset_index(drop=True)
        
    final_df = aggregate_entities(pred_df)
    final_df = final_df[["Age", "Sex", "History", "Sign_symptom", "Medication"]]

    return pred_df, final_df

In [ ]:
texto = (
    "The patient is a 65-year-old male with a history of hypertension and "
    "type 2 diabetes mellitus. He presented with chest pain and shortness "
    "of breath. He was started on aspirin 81 mg daily and metoprolol 25 mg "
    "twice daily. CT scan revealed bilateral pulmonary embolism."
)

ner_prediction(texto)